In [1]:
!pip uninstall -qqy jupyterlab kfp
!pip install -U google-genai chromadb beautifulsoup4 requests tenacity groq cloudscraper nest-asyncio
!pip install PyPDF2 python-docx
!pip install feedparser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 13.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 63.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
from google import genai
from google.genai import types
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry
from google.api_core.exceptions import ServiceUnavailable
import json
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import os
import time
import re
from urllib.parse import urlparse
from tenacity import retry as tenacity_retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from groq import Groq
import cloudscraper
import gradio as gr
import nest_asyncio
import asyncio


# nest_asyncio.apply()  # diperlukan jika pakai Playwright, tapi tidak wajib

print("✅ Semua library berhasil diimport")

✅ Semua library berhasil diimport


In [3]:
from kaggle_secrets import UserSecretsClient
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
print("✅ Gemini dari Kaggle Secrets")

client = genai.Client(api_key=GOOGLE_API_KEY)

✅ Gemini dari Kaggle Secrets


In [4]:
from kaggle_secrets import UserSecretsClient
GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
print("✅ Groq dari Kaggle Secrets")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq siap")

✅ Groq dari Kaggle Secrets
✅ Groq siap


In [5]:
from kaggle_secrets import UserSecretsClient
GNEWS_API_KEY = UserSecretsClient().get_secret("GNEWS_API_KEY")
print("✅ GNews dari Kaggle Secrets")

✅ GNews dari Kaggle Secrets


Fungsi embedding Chroma

In [6]:
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, document_mode: bool = True):
        super().__init__()
        self.document_mode = document_mode

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"
        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(task_type=embedding_task),
        )
        return [e.values for e in response.embeddings]

**Fungsi chunking teks**

In [7]:
# ============================================================
# PERBAIKAN 1: Chunk size lebih besar + overlap antar chunk
# ============================================================
def chunk_text(text, max_chunk_size=900, overlap=150):
    """
    Memecah teks menjadi chunk dengan ukuran lebih besar dan overlap.
    overlap = jumlah karakter yang diulang di chunk berikutnya
    agar konteks tidak terpotong.
    """
    sentences = text.split('. ')
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 2 <= max_chunk_size:
            current += sent + ". "
        else:
            if current:
                chunks.append(current.strip())
            # Ambil bagian akhir current sebagai overlap ke chunk berikutnya
            overlap_text = current[-overlap:] if len(current) > overlap else current
            current = overlap_text + sent + ". "

    if current:
        chunks.append(current.strip())

    return chunks if chunks else [text]

In [8]:
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 37.6 MB/s eta 0:00:00:00:0100:01
170.4 MiB [                    ] 0% 346.2s170.4 MiB [                    ] 0% 22.6s170.4 MiB [                    ] 0% 12.6s170.4 MiB [                    ] 0% 9.5s170.4 MiB [                    ] 1% 4.7s170.4 MiB [=                   ] 2% 3.1s170.4 MiB [=                   ] 3% 2.6s170.4 MiB [=                   ] 3% 2.7s170.4 MiB [=                   ] 5% 2.4s170.4 MiB [=                   ] 6% 2.1s170.4 MiB [==                  ] 7% 2.0s170.4 MiB [==                  ] 9% 1.9s170.4 MiB [==                  ] 10% 1.8s170.4 MiB [==                  ] 11% 1.7s170.4 MiB [==                  ] 12% 1.7s170.4 MiB [===                 ] 13% 1.6s170.4 MiB [===                 ] 14% 1.6s170.4 MiB [===                 ] 15% 1.6s170.4 MiB [===                 ] 16% 1.6s170.4 MiB [===                 ] 17% 1.5s170.4 MiB [====                ] 18% 1.5s170.4 MiB [====                ] 19% 1.5s170.4 MiB [==== 

In [9]:
import cloudscraper
from curl_cffi import requests as cf_requests
import feedparser
import re

def scrape_via_rss(url):
    """Coba ambil konten via RSS feed - support situs global dan Indonesia"""
    from urllib.parse import urlparse
    domain = urlparse(url).netloc.replace("www.", "")

    rss_map = {
        # === Indonesia ===
        "detik.com": "https://rss.detik.com/index.php/detikcom",
        "kompas.com": "https://rss.kompas.com/asset/html/rss/kompascomall.xml",
        "cnnindonesia.com": "https://www.cnnindonesia.com/rss",
        "liputan6.com": "https://www.liputan6.com/rss",
        "tribunnews.com": "https://www.tribunnews.com/rss",
        "antaranews.com": "https://www.antaranews.com/rss/terkini.xml",
        "tempo.co": "https://rss.tempo.co/",
        "okezone.com": "https://sindikasi.okezone.com/index.php/rss/0/XML",
        "bisnis.com": "https://feeds.bisnis.com/bisnis/rss/terkini",
        "cnbcindonesia.com": "https://www.cnbcindonesia.com/rss",
        "merdeka.com": "https://www.merdeka.com/feed/",
        "jpnn.com": "https://www.jpnn.com/rss/terbaru",
        # === Global ===
        "bbc.com": "https://feeds.bbci.co.uk/news/rss.xml",
        "bbc.co.uk": "https://feeds.bbci.co.uk/news/rss.xml",
        "cnn.com": "http://rss.cnn.com/rss/edition.rss",
        "reuters.com": "https://feeds.reuters.com/reuters/topNews",
        "aljazeera.com": "https://www.aljazeera.com/xml/rss/all.xml",
        "theguardian.com": "https://www.theguardian.com/world/rss",
        "nytimes.com": "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
        "washingtonpost.com": "https://feeds.washingtonpost.com/rss/world",
        "apnews.com": "https://rsshub.app/apnews/topics/apf-topnews",
        "bloomberg.com": "https://feeds.bloomberg.com/markets/news.rss",
        "forbes.com": "https://www.forbes.com/real-time/feed2/",
        "techcrunch.com": "https://techcrunch.com/feed/",
        "theverge.com": "https://www.theverge.com/rss/index.xml",
        "wired.com": "https://www.wired.com/feed/rss",
        "ft.com": "https://www.ft.com/rss/home",
        "economist.com": "https://www.economist.com/rss",
        "time.com": "https://time.com/feed/",
        "newsweek.com": "https://www.newsweek.com/rss",
        "nbcnews.com": "https://feeds.nbcnews.com/nbcnews/public/news",
        "foxnews.com": "https://moxie.foxnews.com/google-publisher/world.xml",
        "sky.com": "https://feeds.skynews.com/feeds/rss/world.xml",
        "dw.com": "https://rss.dw.com/xml/rss-en-all",
        "france24.com": "https://www.france24.com/en/rss",
        "scmp.com": "https://www.scmp.com/rss/91/feed",
    }

    rss_url = rss_map.get(domain)
    if not rss_url:
        # Coba tebak RSS URL umum
        for suffix in ['/rss', '/feed', '/rss.xml', '/feed.xml', '/rss/index.xml']:
            rss_url = f"https://{domain}{suffix}"
            try:
                feed = feedparser.parse(rss_url)
                if feed.entries:
                    break
            except:
                continue

    try:
        feed = feedparser.parse(rss_url)
        for entry in feed.entries:
            if entry.link in url or url in entry.link:
                content = entry.get('summary', '') or entry.get('content', [{}])[0].get('value', '')
                soup = BeautifulSoup(content, 'html.parser')
                text = soup.get_text()
                if len(text) >= 100:
                    return ' '.join(text.split())
    except Exception as e:
        print(f"  ⚠️ RSS gagal: {e}")
    return None


def scrape_from_url(url_input):
    raw_url = re.sub(r'^.*?(?:url|link)\s*:\s*', '', url_input.strip(), flags=re.IGNORECASE)
    if not raw_url.startswith(('http://', 'https://')):
        raw_url = 'https://' + raw_url

    print(f"  ⚡ Mengambil konten dari: {raw_url}")

    # --- Strategi 1: RSS Feed ---
    try:
        rss_content = scrape_via_rss(raw_url)
        if rss_content and len(rss_content) >= 100:
            print("  ✅ Berhasil via RSS")
            return rss_content
    except Exception as e:
        print(f"  ⚠️ RSS gagal: {e}")

    # --- Strategi 2: requests dengan headers lengkap ---
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
            "Referer": "https://www.google.com/",
            "DNT": "1",
            "Connection": "keep-alive",
        }
        response = requests.get(raw_url, headers=headers, timeout=20)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
                tag.decompose()
            article = (soup.find('article') or
                      soup.find('main') or
                      soup.find(class_=re.compile(r'content|article|berita|detail|story|post-body', re.I)))
            paragraphs = article.find_all('p') if article else soup.find_all('p')
            text = ' '.join(p.get_text() for p in paragraphs)
            text = ' '.join(text.split())
            if len(text) >= 100:
                print("  ✅ Berhasil dengan requests biasa")
                return text
    except Exception as e:
        print(f"  ⚠️ requests biasa gagal: {e}")

    # --- Strategi 3: cloudscraper ---
    try:
        scraper = cloudscraper.create_scraper(delay=3)
        response = scraper.get(raw_url, timeout=25)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
                tag.decompose()
            article = (soup.find('article') or
                      soup.find('main') or
                      soup.find(class_=re.compile(r'content|article|berita|detail|story|post-body', re.I)))
            paragraphs = article.find_all('p') if article else soup.find_all('p')
            text = ' '.join(p.get_text() for p in paragraphs)
            text = ' '.join(text.split())
            if len(text) >= 100:
                print("  ✅ Berhasil dengan cloudscraper")
                return text
    except Exception as e:
        print(f"  ⚠️ cloudscraper gagal: {e}")

    # --- Strategi 4: curl_cffi dengan berbagai browser ---
    try:
        for browser in ["chrome120", "chrome110", "chrome107", "safari15_5", "safari17_0"]:
            response = cf_requests.get(raw_url, impersonate=browser, timeout=25)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
                    tag.decompose()
                article = (soup.find('article') or
                          soup.find('main') or
                          soup.find(class_=re.compile(r'content|article|berita|detail|story|post-body', re.I)))
                paragraphs = article.find_all('p') if article else soup.find_all('p')
                text = ' '.join(p.get_text() for p in paragraphs)
                text = ' '.join(text.split())
                if len(text) >= 100:
                    print(f"  ✅ Berhasil dengan curl_cffi ({browser})")
                    return text
    except Exception as e:
        print(f"  ⚠️ curl_cffi gagal: {e}")

    return "Gagal mengambil berita setelah mencoba semua metode."

In [10]:
def scrape_tweet_from_url(url_input):
    url = re.sub(r'^.*?(?:url|link)\s*:\s*', '', url_input.strip(), flags=re.IGNORECASE)
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url
    match = re.search(r'status(?:es)?/(\d+)', url)
    if not match:
        return "Gagal: URL Tweet tidak valid."
    tweet_id = match.group(1)
    print(f"  ↳ Mengambil tweet ID: {tweet_id}")
    api_url = f"https://api.fxtwitter.com/status/{tweet_id}"
    try:
        resp = requests.get(api_url, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        txt = data.get('tweet', data.get('text'))
        if isinstance(txt, dict):
            txt = txt.get('text') or txt.get('content') or str(txt)
        return txt or "Tidak ada teks."
    except Exception as e:
        return f"Gagal: {e}"

In [11]:
def search_news_by_keyword(keyword, max_results=10):
    if GNEWS_API_KEY == "YOUR_GNEWS_API_KEY_HERE":
        print("❌ GNews API key belum diisi.")
        return []
    url = "https://gnews.io/api/v4/search"
    params = {
        "q": keyword,
        "lang": "id",
        "country": "id",
        "max": max_results,
        "apikey": GNEWS_API_KEY
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        articles = resp.json().get("articles", [])
        results = []
        for art in articles:
            results.append({
                "title": art["title"],
                "url": art["url"],
                "source": art["source"]["name"],
                "date": art.get("publishedAt", "")
            })
        return results
    except Exception as e:
        print(f"Gagal mencari: {e}")
        return []

In [12]:
# ============================================================
# Sel 11: Inisialisasi database Chroma dan fungsi tambah berita
# ============================================================

# Inisialisasi embedding function
embed_fn = GeminiEmbeddingFunction(document_mode=True)

# ChromaDB persistent (data disimpan di folder "./news_db")
# ✅ PERBAIKAN 5: Simpan di /kaggle/working agar lebih stabil di Kaggle
import os
DB_PATH = "/kaggle/working/news_db"
os.makedirs(DB_PATH, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=DB_PATH)
print(f" ChromaDB disimpan di: {DB_PATH}")
collection = chroma_client.get_or_create_collection(
    name="news_collection",
    embedding_function=embed_fn
)

# Kosongkan database jika ingin memulai baru (opsional, komentari jika tidak ingin)
try:
    existing = collection.get()['ids']
    if existing:
        collection.delete(ids=existing)
        print(" Database lama dibersihkan")
except:
    pass

# Variabel global untuk menyimpan berita terakhir yang ditambahkan
last_added_content = ""
last_added_title = ""
last_added_source = ""
last_added_date = ""

import hashlib  # tambahkan di bagian atas sel ini

def add_news_to_db(title, content, source, date=None):
    """
    Menambah berita ke database vektor.
    PERBAIKAN: Cek duplikat via hash konten sebelum menyimpan.
    """
    global last_added_content, last_added_title

    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")

    #  PERBAIKAN 2: Cek duplikat dengan hash MD5
    content_hash = hashlib.md5(content.encode()).hexdigest()
    existing = collection.get()
    existing_hashes = [m.get("content_hash", "") for m in existing.get("metadatas", [])]

    if content_hash in existing_hashes:
        print(f" Berita '{title}' sudah ada di database, dilewati (duplikat).")
        return

    chunks = chunk_text(content)

    # Cari ID terakhir untuk penomoran
    all_ids = existing['ids']
    last_num = 0
    for id_str in all_ids:
        if id_str.startswith("news_"):
            try:
                num = int(id_str.split("_")[1])
                last_num = max(last_num, num)
            except:
                pass

    # Tambahkan setiap chunk ke database
    for i, chunk in enumerate(chunks):
        doc_id = f"news_{last_num + i + 1}"
        collection.add(
            documents=[chunk],
            metadatas=[{
                "title": title,
                "date": date,
                "source": source,
                "preview": content[:200],
                "content_hash": content_hash  # ✅ simpan hash untuk cek duplikat
            }],
            ids=[doc_id]
        )

    last_added_content = content
    last_added_title = title
    last_added_source = source
    last_added_date = date

    print(f" '{title}' ditambahkan ({len(chunks)} chunk)")

/tmp/ipykernel_57/347908004.py:5: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  super().__init__()


 ChromaDB disimpan di: /kaggle/working/news_db


Retrieval & generate jawaban

In [13]:
def retrieve_news(query, n_results=5):
    """Mencari berita relevan dari database Chroma berdasarkan query"""
    embed_fn.document_mode = False
    results = collection.query(query_texts=[query], n_results=n_results)
    return {
        "documents": results['documents'][0],
        "metadatas": results['metadatas'][0],
        "distances": results['distances'][0],
        "ids": results['ids'][0]
    }

def rerank_results(retrieved, top_k=3):
    combined = list(zip(
        retrieved['documents'],
        retrieved['metadatas'],
        retrieved['distances']
    ))
    combined_sorted = sorted(combined, key=lambda x: x[2])
    top = combined_sorted[:top_k]
    return {
        "documents": [x[0] for x in top],
        "metadatas": [x[1] for x in top],
        "distances": [x[2] for x in top]
    }

# ✅ TAMBAHAN BARU
@tenacity_retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(Exception)
)
def generate_answer_groq(query, retrieved):
    context = ""
    for i, (doc, meta) in enumerate(zip(retrieved['documents'], retrieved['metadatas'])):
        context += f"\n--- Berita {i+1} ---\n"
        context += f"Judul: {meta['title']}\nSumber: {meta['source']}\nTanggal: {meta['date']}\n"
        context += f"Isi: {doc}\n"

    prompt = f"""Anda adalah asisten analis berita. Jawab pertanyaan berdasarkan berita yang disediakan.
Gunakan bahasa Indonesia. Jika informasi tidak cukup, katakan dengan jujur.

Pertanyaan: {query}

{context}

Jawaban ringkas dan informatif:"""

    try:
        print("🔄 Mencoba Groq...")
        chat = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
        )
        print("✅ Groq sukses")
        return chat.choices[0].message.content
    except Exception as e:
        print(f"⚠️ Groq error: {e}, fallback ke Gemini...")
        resp = client.models.generate_content(model="gemini-2.0-flash", contents=prompt)
        return resp.text

In [14]:
def ask_news(query, n_results=5):
    global last_added_content, last_added_title, last_added_source, last_added_date

    # Jika ada berita yang baru diinput, SELALU jawab dari berita itu
    # sampai user input berita baru lagi
    if last_added_content:
        print("🔄 Menjawab dari berita yang baru diinput...")
        fake_retrieved = {
            "documents": [last_added_content],
            "metadatas": [{"title": last_added_title, "source": last_added_source, "date": last_added_date}]
        }
        answer = generate_answer_groq(query, fake_retrieved)
        sources = [{"title": last_added_title, "source": last_added_source, "date": last_added_date}]
        return answer, sources

    # Jika belum ada berita yang diinput, cari dari database
    print("🔄 Mencari dari database...")
    retrieved = retrieve_news(query, n_results)
    retrieved = rerank_results(retrieved, top_k=3)
    answer = generate_answer_groq(query, retrieved)
    sources = [{"title": m['title'], "source": m['source'], "date": m['date']}
               for m in retrieved['metadatas']]
    return answer, sources

In [15]:
def summarize_news_from_url(url):
    print(" Mengambil konten berita...")
    content = scrape_from_url(url)
    if content.startswith("Gagal") or content.startswith("Teks terlalu pendek"):
        return None, content

    prompt = f"Ringkas berita berikut dalam 3-4 kalimat, bahasa Indonesia, fokus pada fakta utama:\n\n{content[:2000]}"

    # ✅ PERBAIKAN 4: Konsisten pakai Groq, fallback ke Gemini jika error
    try:
        resp = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
        )
        summary = resp.choices[0].message.content
        return content, summary
    except Exception as e:
        print(f" Groq gagal: {e}, fallback ke Gemini...")
        try:
            resp = client.models.generate_content(model="gemini-2.0-flash", contents=prompt)
            return content, resp.text
        except Exception as e2:
            return content, f"Gagal meringkas: {e2}"

In [16]:
# ============================================================
# Sel 14: Menu input berita (manual, URL, Twitter, cari keyword, file)
# ============================================================
import os

def read_file_from_path():
    """Membaca konten dari file (txt, json, csv) berdasarkan path yang dimasukkan user"""
    print("\n MEMBACA FILE DARI PATH")
    print("Contoh path: ./berita.txt (jika file di working directory)")
    print("Di Kaggle: /kaggle/input/nama-dataset/berita.txt")
    file_path = input("Masukkan path lengkap file: ").strip()
    if not os.path.exists(file_path):
        print(f" File tidak ditemukan: {file_path}")
        return None, None
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    filename = os.path.basename(file_path)
    print(f" Berhasil membaca {filename} ({len(content)} karakter)")
    return content, filename

def input_news_menu():
    print("\n PILIH SUMBER BERITA:")
    print("1. Masukkan teks manual (copy-paste)")
    print("2. Ambil dari URL (Berita Biasa)")
    print("3. Ambil dari URL X/Twitter")
    print("4. Cari berita berdasarkan kata kunci (contoh: trump, ekonomi)")
    print("5. Baca dari file (txt, json, csv)")
    choice = input("Pilih (1/2/3/4/5): ").strip()
    
    if choice == '1':
        title = input("Judul: ").strip()
        if not title:
            print(" Judul tidak boleh kosong.")
            return
        print("Isi berita (baris kosong untuk selesai):")
        lines = []
        while True:
            line = input()
            if line == "":
                break
            lines.append(line)
        content = "\n".join(lines)
        source = input("Sumber (optional): ").strip() or "Manual"
        add_news_to_db(title, content, source)
    
    elif choice == '2':
        url = input("URL berita: ").strip()
        if not url:
            return
        content = scrape_from_url(url)
        if content.startswith("Gagal") or len(content) < 100:
            print(content)
            return
        print(f"Berhasil, {len(content)} karakter.")
        title = input("Judul (kosongkan untuk otomatis): ").strip()
        if not title:
            from urllib.parse import urlparse
            path = urlparse(url).path
            title = path.split('/')[-1].replace('-', ' ') if path else "Berita dari URL"
        source = input("Sumber (optional): ").strip() or "URL"
        add_news_to_db(title, content, source)
    
    elif choice == '3':
        url = input("URL X/Twitter: ").strip()
        if not url:
            return
        content = scrape_tweet_from_url(url)
        if content.startswith("Gagal"):
            print(content)
            return
        print(f"Berhasil, {len(content)} karakter.")
        title = input("Judul (kosongkan untuk otomatis): ").strip()
        if not title:
            title = f"Tweet: {url.split('/')[-1]}"
        source = "X (Twitter)"
        add_news_to_db(title, content, source)
    
    elif choice == '4':
        keyword = input("Kata kunci pencarian: ").strip()
        if not keyword:
            return
        interactive_search(keyword)   # fungsi ini ada di Sel 15
    
    elif choice == '5':
        content, filename = read_file_from_path()
        if content is None:
            return
        title = input(f"Judul untuk file '{filename}' (kosongkan pakai nama file): ").strip()
        if not title:
            title = filename
        source = input("Sumber (optional): ").strip() or "File"
        add_news_to_db(title, content, source)
    
    else:
        print("Pilihan tidak valid.")

In [17]:
def interactive_search(keyword):
    print(f"\n Mencari berita tentang '{keyword}'...")
    results = search_news_by_keyword(keyword, max_results=10)
    if not results:
        print("Tidak ada hasil.")
        return
    print(f"\n Ditemukan {len(results)} berita:")
    for i, item in enumerate(results, 1):
        print(f"{i}. {item['title'][:80]}")
        print(f"   Sumber: {item['source']} | {item['url']}\n")
    
    while True:
        pilih = input("Pilih nomor berita (atau 'batal'): ").strip()
        if pilih.lower() == 'batal':
            return
        try:
            idx = int(pilih) - 1
            if 0 <= idx < len(results):
                selected = results[idx]
                print(f"\n Dipilih: {selected['title']}")
                
                print("\n Opsi:")
                print("  1. Tampilkan ringkasan berita")
                print("  2. Langsung tanya tentang berita ini")
                print("  3. Batal")
                sub = input("Pilih (1/2/3): ").strip()
                
                if sub == '1':
                    # Ambil konten dan ringkasan
                    content, summary = summarize_news_from_url(selected['url'])
                    if content is None:
                        print(summary)
                        return
                    print("\n RINGKASAN BERITA:")
                    print("="*50)
                    print(summary)
                    print("="*50)
                    
                    # Opsi setelah ringkasan: apakah ingin bertanya?
                    print("\n Apakah Anda ingin bertanya tentang berita ini?")
                    tanya = input("Ketik 'ya' untuk bertanya, atau 'tidak' / Enter untuk selesai: ").strip().lower()
                    if tanya == 'ya':
                        print("\nSilakan tulis pertanyaan Anda:")
                        question = input("🧑 Anda: ")
                        prompt = f"Berita: {content[:1500]}\nPertanyaan: {question}\nJawab berdasarkan berita di atas dalam bahasa Indonesia:"
                        try:
                            resp = groq_client.chat.completions.create(
                                messages=[{"role": "user", "content": prompt}],
                                model="llama-3.3-70b-versatile",
                            )
                            print(f"\n🤖 Jawaban: {resp.choices[0].message.content}")
                        except Exception as e:
                            print(f"Error: {e}")
                    else:
                        print("Terima kasih. Kembali ke menu.")
                    break
                
                elif sub == '2':
                    # Langsung tanya tanpa ringkasan
                    print("\nTanyakan sesuatu tentang berita ini:")
                    question = input("🧑 Anda: ")
                    content = scrape_from_url(selected['url'])
                    if content.startswith("Gagal") or len(content) < 100:
                        print(f"Gagal mengambil konten: {content}")
                        return
                    prompt = f"Berita: {content[:1500]}\nPertanyaan: {question}\nJawab berdasarkan berita di atas dalam bahasa Indonesia:"
                    try:
                        resp = groq_client.chat.completions.create(
                            messages=[{"role": "user", "content": prompt}],
                            model="llama-3.3-70b-versatile",
                        )
                        print(f"\n🤖 Jawaban: {resp.choices[0].message.content}")
                    except Exception as e:
                        print(f"Error: {e}")
                    break
                else:
                    print("Batal.")
                    break
            else:
                print("Nomor tidak valid.")
        except ValueError:
            print("Input tidak valid.")

In [18]:
def chat_news():
    print("\n📰 CHATBOT ANALISIS BERITA (RAG)")
    print("Perintah:")
    print("  - ketik 'tambah'        : masuk menu input berita")
    print("  - ketik 'exit'          : keluar")
    print("  - Selain itu, Anda bisa langsung bertanya tentang berita yang sudah tersimpan.\n")
    while True:
        user_input = input("🧑 Anda: ").strip()
        if user_input.lower() == 'exit':
            print("👋 Sampai jumpa!")
            break
        elif user_input.lower() == 'tambah':
            input_news_menu()
        else:
            try:
                answer, sources = ask_news(user_input)
                print(f"\n🤖 Jawaban: {answer}")
                if sources:
                    src_text = ", ".join([f"{s['title']} ({s['source']})" for s in sources])
                    print(f"📚 Sumber: {src_text}")
                print()
            except Exception as e:
                print(f"⚠️ Error: {e}\n")

In [19]:
def answer_from_gradio(message, history):
    try:
        answer, sources = ask_news(message)
        if sources and not answer.startswith(("Maaf", "Error")):
            src_text = "\n".join([f"- {s['title']} ({s['source']})" for s in sources])
            return f"{answer}\n\n📚 **Sumber:**\n{src_text}"
        else:
            return answer
    except Exception as e:
        return f"⚠️ Error: {e}"

In [20]:
import gradio as gr
import os
import PyPDF2
from docx import Document

# ========== Fungsi untuk bertanya ==========
def answer_from_gradio(message, history):
    try:
        answer, sources = ask_news(message)
        if sources and not answer.startswith(("Maaf", "Error")):
            src_text = "\n".join([f"- {s['title']} ({s['source']})" for s in sources])
            return f"{answer}\n\n📚 **Sumber:**\n{src_text}"
        else:
            return answer
    except Exception as e:
        return f"⚠️ Error: {e}"

# ========== Fungsi menambah berita (manual, URL, Twitter, file) ==========
def add_manual_gradio(title, content, source):
    if not title.strip() or not content.strip():
        return "❌ Judul dan isi harus diisi."
    add_news_to_db(title, content, source)
    return f"✅ Berita '{title}' ditambahkan! Silakan bertanya di chat."

def add_url_gradio(url, source):
    if not url.strip():
        return "❌ URL kosong."
    content = scrape_from_url(url)
    if content.startswith("Gagal") or len(content) < 100:
        return f"❌ Gagal mengambil berita.\n\nSilakan coba salin isi berita ke tab 📝 Teks Manual."
    from urllib.parse import urlparse
    path = urlparse(url).path
    domain = urlparse(url).netloc.replace("www.", "")
    title = path.split('/')[-1].replace('-', ' ') if path else "Berita dari URL"
    add_news_to_db(title, content, source or domain)
    return f"✅ Berhasil! '{title}' disimpan ({len(content)} karakter). Silakan bertanya di chat."

def add_twitter_gradio(url, source):
    if not url.strip():
        return "❌ URL kosong."
    content = scrape_tweet_from_url(url)
    if content.startswith("Gagal"):
        return f"❌ {content}"
    title = f"Tweet: {url.split('/')[-1]}"
    add_news_to_db(title, content, source or "X (Twitter)")  # ✅ lebih deskriptif
    return f"✅ Tweet '{title}' ditambahkan! Silakan bertanya di chat."
    
def add_file_gradio(file_obj, source):
    if file_obj is None:
        return "❌ File kosong."
    filename = file_obj.name
    content = ""
    try:
        if filename.endswith('.pdf'):
            reader = PyPDF2.PdfReader(filename)
            content = " ".join([page.extract_text() for page in reader.pages if page.extract_text()])
        elif filename.endswith('.docx'):
            doc = Document(filename)
            content = "\n".join([para.text for para in doc.paragraphs])
        else:
            with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
        if not content.strip():
            return "❌ File kosong atau tidak bisa dibaca."
        title = os.path.basename(filename)
        add_news_to_db(title, content, source or "File")
        return f"✅ Berita dari file '{title}' ditambahkan! Silakan bertanya di chat."
    except Exception as e:
        return f"❌ Terjadi kesalahan saat membaca file: {e}"

# ========== Fungsi pencarian kata kunci ==========
def update_dropdown_after_search(keyword):
    results = search_news_by_keyword(keyword, max_results=10)
    if not results:
        return "🔍 Tidak ada hasil.", gr.Dropdown(choices=[]), results
    choices = [f"{i+1}. {res['title'][:80]}" for i, res in enumerate(results)]
    return f"🔍 Ditemukan {len(results)} berita. Pilih dari dropdown.", gr.Dropdown(choices=choices), results

# ========== Fungsi simpan dari hasil pencarian ==========
def save_and_confirm(selection, results_state):
    if not selection or not results_state:
        return "❌ Belum ada berita dipilih."
    try:
        idx = int(selection.split('.')[0]) - 1
        if not (0 <= idx < len(results_state)):
            return "❌ Pilihan tidak valid."
        selected = results_state[idx]
        print(f"📥 Mengambil konten: {selected['url']}")
        content = scrape_from_url(selected['url'])
        if content.startswith("Gagal") or len(content) < 100:
            return f"❌ Gagal mengambil konten: {content}"
        add_news_to_db(selected['title'], content, selected['source'])  # ✅ sudah pakai nama sumber asli dari GNews
        return f"✅ Berita '{selected['title']}' berhasil disimpan! Silakan bertanya di chat."
    except Exception as e:
        return f"❌ Error: {e}"

# ========== Membangun antarmuka Gradio ==========
with gr.Blocks(title="RAG News Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("#  RAG News Analyzer")
    gr.Markdown("Sistem analisis berita dengan Retrieval-Augmented Generation (RAG)")

    gr.Markdown("##  Tambah Berita")
    with gr.Tabs():
        with gr.TabItem(" Teks Manual"):
            title_manual = gr.Textbox(label="Judul")
            content_manual = gr.Textbox(label="Isi Berita", lines=8)
            source_manual = gr.Textbox(label="Sumber (opsional)")
            btn_manual = gr.Button("Simpan Berita")
            status_manual = gr.Textbox(label="Status", interactive=False)
            btn_manual.click(add_manual_gradio, [title_manual, content_manual, source_manual], status_manual)

        with gr.TabItem(" URL Berita"):
            url_input = gr.Textbox(label="URL Berita")
            url_source = gr.Textbox(label="Sumber (opsional)")
            btn_url = gr.Button("Ambil & Simpan")
            status_url = gr.Textbox(label="Status", interactive=False)
            btn_url.click(add_url_gradio, [url_input, url_source], status_url)

        with gr.TabItem(" X/Twitter"):
            twitter_input = gr.Textbox(label="URL Tweet")
            twitter_source = gr.Textbox(label="Sumber (opsional)")
            btn_twitter = gr.Button("Ambil & Simpan")
            status_twitter = gr.Textbox(label="Status", interactive=False)
            btn_twitter.click(add_twitter_gradio, [twitter_input, twitter_source], status_twitter)

        with gr.TabItem(" Upload File"):
            file_input = gr.File(label="Pilih file (.txt, .pdf, .docx)", file_types=[".txt", ".pdf", ".docx"])
            file_source = gr.Textbox(label="Sumber (opsional)")
            btn_file = gr.Button("Simpan dari File")
            status_file = gr.Textbox(label="Status", interactive=False)
            btn_file.click(add_file_gradio, [file_input, file_source], status_file)

        with gr.TabItem(" Cari Kata Kunci"):
            gr.Markdown("### Cari berita dari GNews berdasarkan kata kunci")
            keyword_input = gr.Textbox(label="Kata kunci", placeholder="Contoh: ekonomi, trump, teknologi")
            search_btn = gr.Button("Cari")
            search_output = gr.Markdown()
            search_state = gr.State([])
            dropdown_news = gr.Dropdown(label="Pilih berita", choices=[])
            btn_keyword = gr.Button("Ambil & Simpan")        # ← sama seperti tab URL
            status_keyword = gr.Textbox(label="Status", interactive=False)  # ← sama seperti tab URL
            search_btn.click(update_dropdown_after_search, [keyword_input], [search_output, dropdown_news, search_state])
            btn_keyword.click(save_and_confirm, [dropdown_news, search_state], status_keyword)

    gr.Markdown("---")
    gr.Markdown("##  Tanya Berita (dari database)")
    gr.ChatInterface(
        fn=answer_from_gradio,
        description="Sistem akan menjawab berdasarkan berita yang sudah tersimpan di database.",
        examples=["Apa isi berita ini?", "Ringkaskan berita ini."]
    )

if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipykernel_57/297789270.py:98: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="RAG News Analyzer", theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2f377a6f18ae9dc610.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
